# Photon Mosaic: synthetic data walkthrough

Generates a synthetic imaging movie with known ground-truth fluorescence, then runs it
through the full analysis pipeline: fluorescence extraction, ΔF/F, deconvolution, and
neuropil subtraction.

In [ ]:
import numpy as np
import spikeinterface.widgets as sw

import photon_mosaic as pm
import photon_mosaic.widgets as pw

%matplotlib widget

## Generate synthetic imaging data

In [ ]:
rois, imaging, ground_truth = pm.generate_imaging_with_rois(
    num_frames=10000, bleaching_time=600.0, noise_std="poisson", weighted_rois=True, decay_time=0.7, seed=0
)

In [ ]:
rois

In [ ]:
imaging

In [ ]:
w = pw.plot_imaging_series(imaging, backend="ipywidgets", vmax_percentile=99.75)
w.colorbars["imaging"].set_label("Photon counts")
w.figure.tight_layout()  # re-run: the label wasn't there for the widget's own tight_layout() call

In [ ]:
pw.plot_rois(rois, backend="ipywidgets", width_cm=20)

## Create the ROI analyzer

In [ ]:
analyzer = pm.create_roi_analyzer(rois, imaging)

## Extract fluorescence

In [ ]:
fluorescence_ext = analyzer.compute("fluorescence")

In [ ]:
fluorescence = fluorescence_ext.get_data(outputs="recording")
fluorescence

In [ ]:
sw.plot_traces(fluorescence, backend="ipywidgets", time_range=[0, 30])

## Compute ΔF/F

In [ ]:
dff_ext = analyzer.compute("df_over_f", n_jobs=4, method="percentile")
df_over_f = dff_ext.get_data(outputs="recording")
sw.plot_traces(df_over_f, backend="ipywidgets", time_range=[30, 60])

## Deconvolution

In [ ]:
deconv_ext = analyzer.compute("deconvolution", n_jobs=4)

In [ ]:
deconvolved = deconv_ext.get_data(outputs="recording")
sw.plot_traces(deconvolved, backend="ipywidgets", time_range=[30, 60])

### Percentile vs. maximin baseline estimation

An aside comparing the two `df_over_f` baseline-estimation methods -- purely for
display below; deconvolution above already used the percentile method.

In [ ]:
dff_ext_maximin = analyzer.compute("df_over_f", n_jobs=4, method="maximin")

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import Dropdown, interactive_output

t = np.arange(fluorescence_ext.get_data().shape[0]) / imaging.sampling_frequency
fig_methods, ax_methods = plt.subplots(2, 1, figsize=(10, 2.5), sharex=True)


def _plot_dff_methods(roi):
    for a in ax_methods:
        a.clear()
    ax_methods[0].plot(t, fluorescence_ext.get_data()[:, roi], c="gray", lw=0.5, label="F")
    ax_methods[0].plot(t, dff_ext.data["f0"][:, roi], c="C0", alpha=0.7, label=r"$F_0$ (percentile)")
    ax_methods[0].plot(t, dff_ext_maximin.data["f0"][:, roi], c="C1", ls="--", alpha=0.7, label=r"$F_0$ (maximin)")
    ax_methods[0].set_ylabel("Fluorescence")
    ax_methods[0].legend(loc="upper right", fontsize=8)

    dff1 = 100 * dff_ext.get_data()[:, roi]
    dff2 = 100 * dff_ext_maximin.get_data()[:, roi]
    ax_methods[1].axhline(0, ls="--", c="k")
    ax_methods[1].plot(t, dff1, c="C0", lw=0.5, alpha=0.7, label="percentile")
    ax_methods[1].plot(t, dff2, c="C1", lw=0.5, ls="--", alpha=0.7, label="maximin")
    ax_methods[1].set_xlim(t[0], t[-1])
    ax_methods[1].set_ylabel(r"$\Delta F/F$ [%]")
    ax_methods[1].set_xlabel("Time (s)")
    ax_methods[1].legend(loc="upper right", fontsize=8)
    fig_methods.tight_layout()


dff_method_dropdown = Dropdown(options=[int(i) for i in rois.roi_ids], description="ROI:")
dff_method_out = interactive_output(_plot_dff_methods, {"roi": dff_method_dropdown})
display(dff_method_dropdown, dff_method_out)

### Reconstructed movies: raw, extracted, denoised

`traces @ ROIs` paints a per-ROI trace back onto the image via the ROIs' own masks --
each ROI's footprint becomes spatially uniform, at whatever that trace's own value is.
`extracted` (same photon-count units as `raw`) isolates the effect of per-ROI spatial
averaging alone -- still temporally noisy. `FluorescenceNode`'s extraction (see its
docstring) is normalized to reconstruct this painting exactly for non-overlapping ROIs;
these ROIs do overlap (see the crosstalk check below), so it's only an approximation
here. `denoised` adds OASIS's temporal denoising on top; since OASIS runs on ΔF/F (not
raw F), its output is converted back to photon-count units via the fitted baseline
`dff_ext.data["f0"]` before painting, so all three panels stay directly comparable.

In [ ]:
raw_masks = rois.get_roi_image_masks()
masks_flat = raw_masks.reshape(rois.get_num_rois(), -1).astype(np.float32)
video_shape = imaging.get_series(epoch_index=0).shape
fluorescence_movie = (fluorescence_ext.get_data() @ masks_flat).reshape(video_shape)
# deconv_ext.data["denoised"] is on the dF/F scale (OASIS runs on dF/F, not raw F) --
# invert dF/F = (F - F0) / F0 to get back to photon counts before painting onto the movie.
denoised_F = deconv_ext.data["denoised"] * dff_ext.data["f0"] + dff_ext.data["f0"]
denoised_movie = (denoised_F @ masks_flat).reshape(video_shape)
fluorescence_imaging = pm.NumpyImaging(fluorescence_movie, sampling_frequency=imaging.sampling_frequency)
denoised_imaging = pm.NumpyImaging(denoised_movie, sampling_frequency=imaging.sampling_frequency)

w = pw.plot_imaging_series(
    {"Raw": imaging, "Extracted": fluorescence_imaging, "Denoised": denoised_imaging},
    backend="ipywidgets",
    width_cm=10,
    vmax_percentile=99.75,
)
for view in ("Raw", "Extracted", "Denoised"):
    w.colorbars[view].set_label("Photon counts")
w.figure.tight_layout()  # re-run: the labels weren't there for the widget's own tight_layout() call

# Extracted and Denoised are on the same (photon-count) scale, so share one contrast range for
# a fair visual comparison. Raw keeps its own (much wider) range: its shot noise spans most of
# that shared range, so the same clipping would saturate Raw to white and hide its structure.
shared_sample = np.concatenate([fluorescence_movie[:100].ravel(), denoised_movie[:100].ravel()])
shared_vmin, shared_vmax = np.percentile(shared_sample, [2.0, 99.75])
for view in ("Extracted", "Denoised"):
    w.global_vmin[view] = shared_vmin
    w.global_vmax[view] = shared_vmax
    w.images[view].set_clim(shared_vmin, shared_vmax)
    w.colorbars[view].update_normal(w.images[view])


### Denoised vs. ground truth (ΔF/F scale)

Validates recovery accuracy: `denoised` (native ΔF/F scale, no `F0` conversion) compared
directly against the true underlying signal, `ground_truth.clean_traces`. `clean_traces`
is exact by construction, unlike a photon-count reconstruction, which would need the true
per-ROI baseline -- not exposed by the generator. Painting both through the same masks on
the same ΔF/F scale makes the comparison direct.

In [ ]:
denoised_dff_movie = (100 * deconv_ext.data["denoised"] @ masks_flat).reshape(video_shape)
ground_truth_movie = (100 * ground_truth.clean_traces @ masks_flat).reshape(video_shape)
denoised_dff_imaging = pm.NumpyImaging(denoised_dff_movie, sampling_frequency=imaging.sampling_frequency)
ground_truth_imaging = pm.NumpyImaging(ground_truth_movie, sampling_frequency=imaging.sampling_frequency)

w2 = pw.plot_imaging_series(
    {"Denoised": denoised_dff_imaging, "Ground truth": ground_truth_imaging},
    backend="ipywidgets",
    width_cm=10,
    vmax_percentile=99.75,
)
for view in ("Denoised", "Ground truth"):
    w2.colorbars[view].set_label(r"$\Delta F/F$ [%]")
w2.figure.tight_layout()  # re-run: the labels weren't there for the widget's own tight_layout() call

# same units for both views -- share one contrast range for a fair comparison.
shared_sample2 = np.concatenate([denoised_dff_movie[:100].ravel(), ground_truth_movie[:100].ravel()])
shared_vmin2, shared_vmax2 = np.percentile(shared_sample2, [2.0, 99.75])
for view in ("Denoised", "Ground truth"):
    w2.global_vmin[view] = shared_vmin2
    w2.global_vmax[view] = shared_vmax2
    w2.images[view].set_clim(shared_vmin2, shared_vmax2)
    w2.colorbars[view].update_normal(w2.images[view])


**Note:** the cell below checks which of the plotted ROIs share pixels with other ROIs.
Unmixed crosstalk between overlapping ROIs can cause small false transients/events in the
traces shown further down.

In [ ]:
masks = rois.get_roi_image_masks() > 0
for i in rois.roi_ids[:3]:
    overlapping = [int(j) for j in rois.roi_ids if j != i and np.any(masks[i] & masks[j])]
    print(f"ROI {i} overlaps with: {overlapping}")

**Note:** inferred ΔF/F is systematically smaller than ground truth below. The video's
`background` (neuropil, out-of-focus light, dark counts) doesn't scale with each ROI's
F0/expression level and isn't subtracted here, which attenuates recovered ΔF/F (see
`generate_imaging_with_rois`'s `background` docs and the neuropil-subtraction section
further down, which corrects for it).

In [ ]:
from IPython.display import display
from ipywidgets import Dropdown, interactive_output

t = np.arange(dff_ext.get_data().shape[0]) / imaging.sampling_frequency
fig_roi, ax_roi = plt.subplots(3, 1, figsize=(10, 4), sharex=True)


def _plot_roi(roi):
    for a in ax_roi:
        a.clear()
    ax_roi[0].plot(t, fluorescence_ext.get_data()[:, roi], lw=0.5, label="F")
    ax_roi[0].plot(t, dff_ext.data["f0"][:, roi], c="#F0E442", label=r"$F_0$")
    ax_roi[0].set_ylabel("Fluorescence")
    ax_roi[0].legend(loc="upper right", fontsize=8)

    ax_roi[1].plot(t, 100 * dff_ext.get_data()[:, roi], c="gray", lw=0.5, alpha=0.7, label=r"inferred $\Delta F/F$")
    ax_roi[1].plot(t, 100 * deconv_ext.data["denoised"][:, roi], c="C0", lw=0.5, alpha=0.7, label="denoised")
    ax_roi[1].plot(t, 100 * ground_truth.clean_traces[:, roi], c="k", lw=0.5, ls="-.", label="ground truth")
    ax_roi[1].set_ylabel(r"$\Delta F/F$ [%]")
    ax_roi[1].legend(loc="upper right", fontsize=8)

    ax_roi[2].plot(t, deconv_ext.get_data()[:, roi], c="C0", lw=0.5, label="deconvolved")
    ax_roi[2].set_xlim(t[0], t[-1])
    spike_times = t[ground_truth.spikes[:, roi] > 0]
    ax_roi[2].vlines(
        spike_times, 0.9, 1.0, transform=ax_roi[2].get_xaxis_transform(), color="k", label="true spikes"
    )
    ax_roi[2].set_ylabel("Activity [a.u.]")
    ax_roi[2].set_xlabel("Time (s)")
    ax_roi[2].legend(loc="upper right", fontsize=8)
    fig_roi.tight_layout()


roi_dropdown = Dropdown(options=[int(i) for i in rois.roi_ids], description="ROI:")
out = interactive_output(_plot_roi, {"roi": roi_dropdown})
display(roi_dropdown, out)

## Neuropil subtraction

Addresses the attenuation noted above: subtracting each ROI's neuropil estimate removes
most of the `background` term from its fluorescence trace before ΔF/F normalization.

`NeuropilExtension` offers two estimates. Both feed the same
`F - neuropil_weight * Fneu` subtraction, on the same scale, so `neuropil_weight` means
the same thing whichever you pick:

| `method` | Background model | Needs | Cost |
| --- | --- | --- | --- |
| `"surround"` | Suite2p-style ring: a *fixed spatial mask* per ROI -- the pixels surrounding it, with every ROI's pixels excluded, weighted `1 / n_ring_pixels` so the extraction reproduces suite2p's unweighted-mean `Fneu`. | `suite2p`; enough non-ROI pixels around each ROI | folded into the `fluorescence` pass |
| `"cnmf"` | CNMF-style low-rank background: fits `Y ≈ C Aᵀ + f bᵀ` with the ROI footprints `A` held **fixed** at the masks, giving `gnb` spatial components `b` with their own timecourses `f`, plus demixed traces `C`. | the movie and the masks only | two streaming passes over the movie (≈2× `fluorescence`) |

In [ ]:
# --- method="surround": Suite2p-style ring mask -------------------------------
analyzer.compute("neuropil", method="surround")
# neuropil_weight=1.0 (full subtraction) rather than the library default of 0.7 (suite2p's
# neucoeff convention): this generator's background is spatially uniform and deterministic,
# so nothing is lost by removing all of it here -- makes the demo's correction as clear as possible.
fluorescence_ext_surround = analyzer.compute("fluorescence", n_jobs=4, use_neuropil=True, neuropil_weight=1.0)
dff_ext_surround = analyzer.compute("df_over_f", n_jobs=4, method="percentile")
deconv_ext_surround = analyzer.compute("deconvolution", n_jobs=4)

# (n_rois, H, W) sparse ring masks. Each ROI's ring sums to 1.0, so extracting through them
# is an unweighted ring *mean* -- suite2p's own Fneu convention. Densified as float32 (the
# movie's dtype) so the projection below stays in float32 rather than upcasting the movie.
ring_masks = analyzer.get_extension("neuropil").get_data()
ring_flat = np.asarray(ring_masks.reshape((rois.get_num_rois(), -1)).todense(), dtype=np.float32)
print("ring masks:", type(ring_masks).__name__, ring_masks.shape)
print("ring pixels per ROI:", (ring_flat > 0).sum(axis=1))
print("ring weight sums (1.0 = a plain mean; 0.0 = no ring found):", ring_flat.sum(axis=1)[:5], "...")

### `method="cnmf"`: low-rank background

An analyzer only ever holds *one* `neuropil` extension, so computing the CNMF background
**replaces** the surround one. `fluorescence` reads whichever neuropil is current at the
moment it runs, and it declares `neuropil` as an *optional* dependency -- which means
swapping the method does **not** auto-invalidate the extensions downstream of it, unlike
a required dependency. Recompute `fluorescence` (and its own children) yourself after the
swap, as below, or the previous method's traces silently stay in place.

That still leaves the comparison further down intact: `compute()` returns a fresh
extension object each time rather than mutating the old one, and this analyzer is
in-memory (`format="memory"`), so the `*_surround` handles above keep their own data even
once the analyzer itself has moved on to CNMF.

`gnb=1` is the right rank here (the background really is rank-1) and is also the cheapest
and most stable choice -- with a 1×1 Gram the non-negativity projection is exact and the
fit deterministic. Bump it to 2-3 for spatially structured background. Note that
`highpass_sigma` is **in pixels**: the default of 20 suits these 5-15 px-radius ROIs, but
rescale it if your pixel size differs much.

In [ ]:
# --- method="cnmf": low-rank background (replaces the surround extension above) ---
neuropil_ext_cnmf = analyzer.compute("neuropil", method="cnmf", gnb=1, n_jobs=4)
fluorescence_ext_cnmf = analyzer.compute("fluorescence", n_jobs=4, use_neuropil=True, neuropil_weight=1.0)
dff_ext_cnmf = analyzer.compute("df_over_f", n_jobs=4, method="percentile")
deconv_ext_cnmf = analyzer.compute("deconvolution", n_jobs=4)

info = neuropil_ext_cnmf.get_data("fit_info")
print(f"gnb={info['gnb']}, converged={info['converged']} after {info['n_iter']} iterations")
# Expect a *large* unexplained fraction here and don't read it as a bad fit: at 0.2 photons/
# pixel/frame under noise_std="poisson", most of ||Y||^2 is shot noise, which no low-rank
# model can or should absorb. cond(...) are the diagnostics to watch instead.
print(f"unexplained variance (residual / ||Y||^2) = {info['objective_full'] / info['ynorm_sq']:.4f}")
print(f"cond(A.T A) = {info['cond_gram_masks']:.3g}, cond(D.T D) = {info['cond_gram_joint']:.3g}")

#### The fitted background field

`get_background()` returns the two factors: `b`, shape `(gnb, H, W, n_planes)` and
L2-normalized per component, and `f`, shape `(gnb, n_frames)`, carrying the scale. The
modelled background movie at frame `t` is `sum_k f[k, t] * b[k]` -- something a ring mask
never gives you. Here the truth is a flat field times `exp(-t / bleaching_time)`, so `b`
should come out close to featureless and `f` should roughly trace the bleaching curve.
Expect a few percent of spatial ripple in `b` and a visibly imperfect correlation for `f`:
the gauge freedom below means ROI activity can load onto the background at no cost to the
fit, and at a background of 0.2 photons/pixel/frame the ROIs dominate the movie's variance.

The two methods' per-ROI neuropil traces are directly comparable, which is what makes
`neuropil_weight` mean the same thing for both: `"cnmf"` stores `neuropil_traces` (the
fitted background projected through each ROI's own L1-normalized mask), and the surround
equivalent is the ring mean, `movie @ ring_maskᵀ`. Watch the last printout: the CNMF trace
should track the true background *more closely* than the ring mean, because it pools all
65536 pixels of the frame into one estimate while each ring averages only its own few
hundred -- visible above as the much noisier orange trace.

In [ ]:
b, f = neuropil_ext_cnmf.get_background()
print("b:", b.shape, " f:", f.shape)

# Ground truth: the generator's `background` scaled by exp(-t / bleaching_time), spatially
# flat -- i.e. exactly rank-1. bleaching_time=600.0 and background=0.2 (its default) above.
bleach = np.exp(-np.arange(len(t)) / (600.0 * imaging.sampling_frequency))
background_true = 0.2 * bleach

fneu_cnmf = neuropil_ext_cnmf.get_data("neuropil_traces")
# The surround equivalent: the ring mean per frame, on the same scale as cnmf's traces.
# float32 @ float32, so this allocates only the (n_frames, n_rois) result, not a movie-sized cast.
fneu_surround = imaging.get_series(epoch_index=0).reshape(len(t), -1) @ ring_flat.T

fig_bg, ax_bg = plt.subplots(1, 3, figsize=(12, 2.6))

im = ax_bg[0].imshow(b[0, :, :, 0])
ax_bg[0].set_title(f"$b$ (spatial): rel. std {b.std() / b.mean():.1%}")
plt.colorbar(im, ax=ax_bg[0], fraction=0.046)

# b is L2-normalised and f carries the scale, so compare f's *shape* by rescaling onto the truth.
f_scaled = f[0] * ((f[0] @ bleach) / (f[0] @ f[0]))
ax_bg[1].plot(t, bleach, c="k", ls="-.", label="true bleaching")
ax_bg[1].plot(t, f_scaled, c="C0", lw=0.8, alpha=0.8, label="$f$ (rescaled)")
ax_bg[1].set_title(f"$f$ (temporal): corr = {np.corrcoef(f[0], bleach)[0, 1]:.4f}")
ax_bg[1].set_xlabel("Time (s)")
ax_bg[1].legend(loc="upper right", fontsize=8)

ax_bg[2].plot(t, fneu_surround[:, 11], c="C1", lw=0.5, alpha=0.7, label='"surround" ring mean')
ax_bg[2].plot(t, fneu_cnmf[:, 11], c="C0", lw=1.0, label=r'"cnmf" $b\,f$ through mask')
ax_bg[2].plot(t, background_true, c="k", ls="-.", label="true background")
ax_bg[2].set_title("$F_{neu}$ for ROI 11")
ax_bg[2].set_xlabel("Time (s)")
ax_bg[2].set_ylabel("Photon counts")
ax_bg[2].legend(loc="upper right", fontsize=8)

fig_bg.tight_layout()

for name, fneu in (("surround", fneu_surround), ("cnmf", fneu_cnmf)):
    err = np.abs(fneu - background_true[:, np.newaxis]).mean()
    print(f"{name:>8}: mean |Fneu - true background| = {err:.4f} photons/pixel/frame")

#### Three-way comparison

`fit_scale` returns the multiplier that best matches each inferred trace to its ground
truth, so a value **above** 1.0 means the recovered ΔF/F is attenuated and has to be
scaled up. Both corrections pull it substantially toward 1.0; neither is expected to land
exactly on it -- with `neuropil_weight=1.0` and a background this close to the shot-noise
floor the correction can overshoot slightly. Both are chasing the same rank-1 background
here, so they end up in much the same place. The point is that they are interchangeable at
the call site, not that either wins on this synthetic movie.

One caveat specific to `"cnmf"`, spelled out in `NeuropilExtension`'s docstring: the
trace/background split has an **exact gauge freedom**. For any `alpha`, taking
`b -> b + A alpha` together with `C -> C - f alphaᵀ` leaves `C Aᵀ + f bᵀ` bit-for-bit
unchanged. What that costs is precisely `b` on ROI-support pixels and each trace's
component along `f`: a trace's absolute offset and slow `f`-shaped drift are not pinned
down, while `f`, the background away from ROIs, and each trace with the `f` direction
projected out all are. Non-negativity narrows the family but does not collapse it, which
is exactly why `neuropil_weight` remains a user knob rather than being fixed at 1.

In [ ]:
def fit_scale(inferred, truth):
    return (inferred * truth).sum(axis=0) / (inferred * inferred).sum(axis=0)


runs = {
    "none": (dff_ext, deconv_ext),
    "surround": (dff_ext_surround, deconv_ext_surround),
    "cnmf": (dff_ext_cnmf, deconv_ext_cnmf),
}
print(f"{'neuropil':>10}  {'median dF/F scale':>18}  {'median deconv scale':>20}   (1.0 = no attenuation)")
for name, (dff_run, deconv_run) in runs.items():
    scale_dff = np.median(fit_scale(dff_run.get_data(), ground_truth.clean_traces))
    scale_deconv = np.median(fit_scale(deconv_run.get_data(), ground_truth.spikes))
    print(f"{name:>10}  {scale_dff:>18.3f}  {scale_deconv:>20.3f}")

from IPython.display import display
from ipywidgets import Dropdown, interactive_output

fig_neuropil, ax_neuropil = plt.subplots(figsize=(10, 2))


def _plot_neuropil_roi(roi):
    ax_neuropil.clear()
    ax_neuropil.plot(t, 100 * dff_ext.get_data()[:, roi], lw=0.5, alpha=0.6, label="no neuropil")
    ax_neuropil.plot(t, 100 * dff_ext_surround.get_data()[:, roi], lw=0.5, alpha=0.8, label='"surround"')
    ax_neuropil.plot(t, 100 * dff_ext_cnmf.get_data()[:, roi], lw=0.5, alpha=0.8, label='"cnmf"')
    ax_neuropil.plot(t, 100 * ground_truth.clean_traces[:, roi], ls="-.", c="k", label="ground truth")
    ax_neuropil.set_xlim(30, 60)
    ax_neuropil.set_ylabel(r"$\Delta F/F$ [%]")
    ax_neuropil.set_xlabel("Time (s)")
    ax_neuropil.legend(loc="upper right", fontsize=8, ncol=2)
    fig_neuropil.tight_layout()


# default to ROI 11: lowest baseline (F0), no mask overlap -- the clearest single example of
# the neuropil correction. Other ROIs (e.g. 1, which overlaps ROIs 7/14) are still browsable.
neuropil_roi_dropdown = Dropdown(options=[int(i) for i in rois.roi_ids], value=11, description="ROI:")
neuropil_roi_out = interactive_output(_plot_neuropil_roi, {"roi": neuropil_roi_dropdown})
display(neuropil_roi_dropdown, neuropil_roi_out)